# Live verification of the chameleon-wing gold answers

**What this notebook is for.** `benchmark/chameleon_bench/` ships 40 items whose gold
answers are already checked two ways: every gold passes 100% of its own AST checkers
(the admission gate, decision D06), and 31 of them are *executed* against a stub backed
by a real Blazar capture. Neither of those proves the golds work on the actual testbed —
a stub is a model of python-chi, and if the model is wrong the gold can share the error.

This notebook closes that gap. It reserves real hardware on Chameleon, runs the gold
code against it, and records what happened.

| | |
|---|---|
| project | **CHI-231225** |
| site | **CHI@TACC** (CHI@UC had 5 free hosts of 132 at capture time) |
| python-chi | 1.2.10 |
| corpus pin | artifacts as fetched August 2026; capture 2026-09-04 |

**Safety rules this notebook follows.**

1. Section 1 refuses to continue unless the token is scoped to CHI-231225.
2. Every reservation cell is separate and marked ⚠️. Nothing reserves on "Run All".
3. Every test tears down in a `finally`. Section 6 sweeps anything left behind.
4. Cheap CPU node types first. `compute_haswell_ib` and `compute_cascadelake` had the
   most free capacity at capture time.

**Caveat, stated up front.** This makes the golds true *as of the run date*, not true
forever. Images get retired and node types change. That is the same caveat the corpus
already carries about being pinned to August 2026 clones, and it belongs in any writeup.


## 1. Authenticate, and refuse to run against the wrong project

An app credential carries its own project scope — there is no `OS_PROJECT_NAME` to set —
so the only way to know which allocation we are about to spend is to ask Keystone. The
earlier credential in this workspace authenticated to CHI-261560, which is exactly the
mistake this cell exists to make impossible.


In [ ]:
import os, json, subprocess, urllib.request
from pathlib import Path

WORKSPACE = Path.cwd().parent.parent          # chameleon-work/
RC = WORKSPACE / 'credentials' / 'app-cred-vivek_tacc_2-openrc.sh'
EXPECT_PROJECT = 'CHI-231225'

# Source the openrc in a subshell with stdin closed; never print the secret.
out = subprocess.run(['bash','-c', f'set -a; . "{RC}" >/dev/null 2>&1; set +a; env'],
                     capture_output=True, text=True, stdin=subprocess.DEVNULL)
for line in out.stdout.splitlines():
    if line.startswith('OS_'):
        k, _, v = line.partition('='); os.environ[k] = v

body = {'auth': {'identity': {'methods': ['application_credential'],
        'application_credential': {'id': os.environ['OS_APPLICATION_CREDENTIAL_ID'],
                                   'secret': os.environ['OS_APPLICATION_CREDENTIAL_SECRET']}}}}
req = urllib.request.Request(os.environ['OS_AUTH_URL'].rstrip('/') + '/auth/tokens',
                             data=json.dumps(body).encode(),
                             headers={'Content-Type': 'application/json'})
with urllib.request.urlopen(req, timeout=30) as r:
    TOKEN = json.load(r)['token']

PROJECT = TOKEN['project']['name']
print('site        :', os.environ['OS_REGION_NAME'])
print('project     :', PROJECT)
print('roles       :', [x['name'] for x in TOKEN['roles']])
print('expires     :', TOKEN['expires_at'])

assert PROJECT == EXPECT_PROJECT, (
    f'REFUSING TO CONTINUE: token is scoped to {PROJECT!r}, not {EXPECT_PROJECT!r}. '
    'Reserving here would draw down the wrong allocation.')
print(f'\nOK — scoped to {PROJECT}. Safe to continue.')


## 2. Read-only inventory: does the recorded capture still describe reality?

Nothing is reserved here. This compares live Blazar against the snapshot the benchmark
grades against, so that if a node type has disappeared we find out before a gold fails
for a reason that has nothing to do with the answer.


In [ ]:
import chi
from chi import hardware

chi.use_site('CHI@TACC')

# NOTE: do NOT call chi.set('project_name', ...) here. An application
# credential already carries its project scope, and asking python-chi to
# request one makes Keystone answer:
#   'Application credentials cannot request a scope. (HTTP 401)'
# The first run of this notebook failed exactly this way.

live_types = set(hardware.get_node_types())
snap = json.loads(Path('../chameleon_bench/data/snapshots/baremetal_tacc_2026-09-04.json').read_text())
snap_totals = snap['_meta']['node_type_totals']

missing = sorted(set(snap_totals) - live_types)
added   = sorted(live_types - set(snap_totals))
print(f'live node types      : {len(live_types)}')
print(f'captured node types  : {len(snap_totals)}')
print(f'in capture, not live : {missing or "none"}')
print(f'live, not in capture : {added or "none"}')

import yaml, glob
needed = set()
for p_ in glob.glob('../chameleon_bench/data/items/*.yaml'):
    for r in (yaml.safe_load(open(p_)).get('exec_expect') or {}).get('node_reservations', []):
        needed.add(r['node_type'])
# Site-aware: an item that reserves at CHI@UC cannot be checked from a
# CHI@TACC token, and reporting it as 'not live' would be a false alarm.
tacc_items = set()
for p_ in glob.glob('../chameleon_bench/data/items/*.yaml'):
    it = yaml.safe_load(open(p_)); ex = it.get('exec_expect') or {}
    if ex.get('site') != 'CHI@TACC':
        continue
    for r in ex.get('node_reservations', []):
        tacc_items.add((it['id'], r['node_type']))
print('\nnode types CHI@TACC items require:')
gone = []
for iid, t in sorted(tacc_items):
    ok = t in live_types
    if not ok: gone.append((iid, t))
    print(f'  {iid:<6} {t:<26} live={ok}')
if gone:
    print(f'\n*** {len(gone)} item(s) name hardware no longer present at CHI@TACC: {gone}')
    print('    These would fail on the live testbed and need retargeting.')


## 3. Pick the cheapest node type that is actually free

Free capacity moves. Rather than hardcode a type, ask Blazar now and prefer the CPU
types the items use — GPU nodes cost more SUs and the binding semantics we are testing
are identical on either.


In [ ]:
TEST_NODE_TYPE = None
PREFERRED = ['compute_haswell_ib', 'compute_cascadelake', 'compute_skylake',
             'compute_cascadelake_r', 'compute_zen3']

# chi.hardware.Node exposes `.type` (and `.name`). `node_type` is the filter
# kwarg on get_nodes(), not a field on the result.
nodes = hardware.get_nodes()
counts = {}
for n in nodes:
    c_ = counts.setdefault(n.type, [0, 0])
    c_[1] += 1
    if n.reservable:
        c_[0] += 1

for t in PREFERRED:
    f, tot = counts.get(t, (0, 0))
    print(f'  {t:<26} {f:>3} reservable / {tot:>3}')

TEST_NODE_TYPE = next((t for t in PREFERRED if counts.get(t, (0, 0))[0] > 0), None)
assert TEST_NODE_TYPE, 'no preferred CPU node type is reservable right now'
print(f'\nusing: {TEST_NODE_TYPE}')

# Server.submit() needs a keypair: Server.__init__ falls back to
# update_keypair(), which returns None when no public key is configured, and
# submit() then dies on self.keypair.name. A machine with no ~/.ssh/*.pub must
# point python-chi at the public half of the key named by $CHI_SSH_KEY. The
# private key is never read into this notebook: ssh-keygen -y derives the
# public half in a subprocess, and only that half is ever written out.
import os, tempfile, subprocess as _sp
KEYFILE = Path(os.environ["CHI_SSH_KEY"]).expanduser()
assert KEYFILE.exists(), f"CHI_SSH_KEY does not exist: {KEYFILE}"
PUBKEY = Path(os.environ.get("CHI_SSH_PUBKEY",
                             Path(tempfile.gettempdir()) / "chi_keypair.pub"))
if not PUBKEY.exists():
    PUBKEY.write_text(_sp.run(["ssh-keygen", "-y", "-f", str(KEYFILE)],
                              capture_output=True, text=True, check=True).stdout)
chi.set("keypair_public_key", str(PUBKEY))
chi.set("keypair_name", os.environ.get("CHI_KEYPAIR_NAME", "goldtest"))
print(f"keypair configured from $CHI_SSH_KEY (public half only) -> {PUBKEY}")


## 4. Teardown helpers

Defined before anything is created, so a failed reservation cell still has something to
clean up with. SUs are not the risk here — a node-hour is about 1 SU against a 20,000 SU
allocation. A lease nobody deletes is the risk.


In [ ]:
from chi import lease as chi_lease
import datetime, traceback

CREATED = {'leases': [], 'servers': []}   # everything this notebook makes
RESULTS = []                              # one row per test

def record(name, passed, detail=''):
    RESULTS.append({'test': name, 'result': 'PASS' if passed else 'FAIL', 'detail': detail})
    print(f"{'PASS' if passed else 'FAIL'}  {name}" + (f'  — {detail}' if detail else ''))

def cleanup():
    """Delete everything this notebook created. Safe to run repeatedly."""
    for s in CREATED['servers']:
        try: s.delete(); print(f'  deleted server {s.name}')
        except Exception as e: print(f'  server {getattr(s,"name","?")}: {e}')
    for l in CREATED['leases']:
        try: l.delete(); print(f'  deleted lease {l.name}')
        except Exception as e: print(f'  lease {getattr(l,"name","?")}: {e}')
    CREATED['servers'].clear(); CREATED['leases'].clear()

def require(**names):
    """Abort a test whose inputs are missing.

    The first run of this notebook recorded PASS on the trap test when what
    actually happened was a NameError: a bare `except Exception` cannot tell
    'Chameleon rejected this' from 'our own code broke', and it scored the
    second as the first. A negative test that passes for the wrong reason is
    worse than no test, so missing inputs now raise before anything is tried.
    """
    missing = [k for k, v in names.items() if v is None]
    if missing:
        raise RuntimeError(f'precondition not met: {missing} is None - an '
                           'earlier cell did not complete. Not recording a result.')

RESERVATION_ID = LEASE_ID = None
s = None

print('helpers ready; nothing reserved yet')


## 5. ⚠️ Reservation tests — each of these SPENDS SUs

Run these one at a time, with permission. Each asserts something a gold depends on.


### 5.1 ⚠️ Does `node_reservations[0]['id']` exist after submit?

**Four items depend on this** (CB01, CB02, CB06, CB09 and every composition item using
the same wiring). Local introspection confirmed `node_reservations` is a real attribute
initialised to `[]`; what it cannot confirm is whether the entries carry an `id` once
Blazar has accepted the lease. That is what this cell settles.


In [ ]:
require(TEST_NODE_TYPE=TEST_NODE_TYPE)

LEASE_HOURS = 1

l = None
try:
    l = chi_lease.Lease(f'goldtest-wiring-{datetime.datetime.now():%H%M%S}',
                        duration=datetime.timedelta(hours=LEASE_HOURS))
    l.add_node_reservation(amount=1, node_type=TEST_NODE_TYPE)
    l.submit(idempotent=True, wait_for_active=True)
    CREATED['leases'].append(l)

    print('lease id     :', l.id)
    print('reservations :', json.dumps(l.node_reservations, indent=1, default=str))

    has = bool(l.node_reservations) and 'id' in l.node_reservations[0]
    record('node_reservations[0]["id"] is populated after submit', has,
           str(l.node_reservations[0].get('id')) if has else 'absent')
    RESERVATION_ID = l.node_reservations[0]['id'] if has else None
    LEASE_ID = l.id
    record('reservation id differs from lease id', RESERVATION_ID != LEASE_ID,
           f'{RESERVATION_ID} vs {LEASE_ID}')
except Exception as e:
    record('lease submit', False, f'{type(e).__name__}: {e}')
    traceback.print_exc()


### 5.2 ⚠️ Does `get_node_reservation` agree with the indexed form?

Golds use both spellings and the checkers accept either. If they disagree, one family of
items is wrong.


In [ ]:
require(LEASE_ID=LEASE_ID, RESERVATION_ID=RESERVATION_ID)

# lease.get_node_reservation is marked '.. deprecated:: 1.0'. In python-chi
# 1.2.10 it also appears BROKEN: _reservation_matching expects a lease dict,
# but get_lease now returns a Lease object, so it raises
#   AttributeError: 'Lease' object has no attribute 'get'
# This matters beyond our golds: the idiom appears throughout the corpus
# (A9, A55, A61, A63, A77 ...), so artifacts are teaching something that no
# longer runs. Recorded as a finding either way.
try:
    via_fn = chi_lease.get_node_reservation(LEASE_ID)
    record('deprecated get_node_reservation still works', via_fn == RESERVATION_ID,
           f'{via_fn} vs {RESERVATION_ID}')
except Exception as e:
    record('deprecated get_node_reservation still works', False,
           f'BROKEN in python-chi 1.2.10 - {type(e).__name__}: {str(e)[:90]}')


### 5.3 ⚠️ Launch a server bound to the reservation

Confirms the image name is still valid at this site and that a correctly-wired instance
actually boots. This is the expensive cell — it holds a real node.


In [ ]:
require(RESERVATION_ID=RESERVATION_ID)

from chi import server as chi_server

IMAGE = 'CC-Ubuntu22.04'   # the image most items assert

s = None
try:
    s = chi_server.Server(f'goldtest-{datetime.datetime.now():%H%M%S}',
                          image_name=IMAGE, reservation_id=RESERVATION_ID)
    s.submit(idempotent=True, wait_for_active=True)
    CREATED['servers'].append(s)
    record(f'server boots on {TEST_NODE_TYPE} with {IMAGE}', s.status == 'ACTIVE', s.status)
except Exception as e:
    record('server launch', False, f'{type(e).__name__}: {e}')
    traceback.print_exc()


### 5.4 ⚠️ The documented trap: pass the LEASE id where a RESERVATION id belongs

Four items and the execution stub assert this fails. The corpus documents it (A55's
`traps_illustrated`) but the corpus is not the testbed. **This is the single most valuable
cell in the notebook**: if it turns out to succeed, our stub is wrong and several items
are testing something that is not true.


In [ ]:
require(LEASE_ID=LEASE_ID)          # abort rather than score a NameError as a pass

bad = None
try:
    bad = chi_server.Server(f'goldtest-trap-{datetime.datetime.now():%H%M%S}',
                            image_name=IMAGE, reservation_id=LEASE_ID)
    bad.submit(idempotent=True, wait_for_active=True)
    CREATED['servers'].append(bad)
    record('lease-id-as-reservation-id is REJECTED', False,
           f'ACCEPTED, reached {bad.status} - the stub and 4 items assume otherwise')
except NameError as e:
    # our bug, not a finding
    raise RuntimeError(f'test could not run: {e}') from e
except Exception as e:
    record('lease-id-as-reservation-id is REJECTED', True,
           f'{type(e).__name__}: {str(e)[:150]}')


### 5.5 ⚠️ Floating IP on the same lease

Several items reserve a node and a floating IP on one lease and then attach the IP.


In [ ]:
require(s=s)

try:
    fip = s.associate_floating_ip()
    record('floating IP attaches to the instance', bool(fip), str(fip))
except Exception as e:
    record('floating IP', False, f'{type(e).__name__}: {e}')


## 6. Teardown — always run this


In [ ]:
cleanup()


### 6.1 Sweep: anything left in the project that we made

Catches leases orphaned by a crashed cell. Only deletes names starting `goldtest-`, so it
can never touch someone else's work in this project.


In [ ]:
for lz in chi_lease.list_leases():
    name = lz['name'] if isinstance(lz, dict) else getattr(lz, 'name', '')
    lid  = lz['id']   if isinstance(lz, dict) else getattr(lz, 'id', '')
    if str(name).startswith('goldtest-'):
        print('orphan ->', name, lid)
        chi_lease.delete_lease(lid)
        print('   deleted')
print('sweep complete')


## 7. Results


In [ ]:
import datetime as _dt
print(f'{"result":<6} {"test":<58} detail')
print('-' * 100)
for r in RESULTS:
    print(f"{r['result']:<6} {r['test']:<58} {r['detail'][:40]}")
print()
passed = sum(1 for r in RESULTS if r['result'] == 'PASS')
print(f'{passed}/{len(RESULTS)} live checks passed')
print(f'project {PROJECT} · site CHI@TACC · node {TEST_NODE_TYPE} · {_dt.datetime.utcnow():%Y-%m-%d %H:%M} UTC')
Path('live_verification_results.json').write_text(json.dumps(
    {'project': PROJECT, 'site': 'CHI@TACC', 'node_type': TEST_NODE_TYPE,
     'utc': _dt.datetime.utcnow().isoformat(), 'results': RESULTS}, indent=1))
print('written: live_verification_results.json')


## 8. Beyond idioms — running the golds themselves

Sections 1–7 verify API *idioms*. They do not answer "does this benchmark's own
gold answer work against the testbed", which is the V2 claim `verification_level`
was always supposed to carry.

The assertions below live in `live_api_probe.py` and `live_gold_batch.py`
rather than in cells, for one reason: a notebook runs all-or-nothing from a
shell, so an operator who is a *process* cannot stop between two spending
cells. Both modules are importable, so there is exactly one implementation and
this notebook is a view onto it rather than a second copy to drift against.


### 8.1 What python-chi needs that a notebook supplies for free

Two defaults the golds inherited from the notebook artifacts they were authored
from, both found only by running gold code *outside* Jupyter (D66):

* `Server(...)` without `key_name` falls through to `update_keypair()`, which
  wants `f"{USER}-jupyter"` and a `keypair_public_key` path from context.
* `Server.submit()` defaults `show="widget"`, and `chi/server.py:391` renders a
  widget only when `_is_ipynb()` is true — **raising otherwise, after the
  instance is already ACTIVE**.

Inside this notebook both are satisfied, which is exactly why neither was ever
visible from here. That is the finding, not a footnote to it.

In [ ]:
import live_api_probe as P

P.authenticate()          # asserts the project before anything can be spent
P.use_site()
print('python-chi', P.chi_version(), '| _is_ipynb here:',
      __import__('chi.server', fromlist=['x'])._is_ipynb())

### 8.2 ⚠️ The full idiom sequence

Each of these is one step of `live_api_probe.py`; run them one at a time. The
teardown registry is on disk, so `teardown` cleans up even if a cell above it
raised or the kernel died.

In [ ]:
st = P.load_state()
P.step_images(st)      # free — every image name the corpus asserts

In [ ]:
P.step_lease(st)       # ⚠️ SPENDS — one lease, the RB grammar
P.step_idioms(st)      # free — both reservation-id spellings

In [ ]:
P.step_boot(st)        # ⚠️ SPENDS — boots one instance
P.step_trap(st)        # ⚠️ SPENDS — the lease-id-as-reservation-id trap

In [ ]:
P.step_teardown(st)    # always safe, always run this
P.step_report(st)

### 8.3 ⚠️ Five golds at once, against the real testbed

`live_gold_batch.py` executes gold answers *verbatim* — written to a file and
run with `runpy` as `__main__`, so what executes is the benchmark's own text —
then asks the API what got created. Parallel because a bare-metal boot is ten
to twenty minutes of waiting and SUs are charged per node-hour, not per
wall-hour: five at once costs the same as five in sequence and takes a fifth of
the time.

Selection is by what is *free right now*; a gold naming fully-booked hardware
is reported SKIPPED, because failing it would grade the calendar. Start with
`--plan`, which costs nothing.

In [ ]:
!python live_gold_batch.py --plan

In [ ]:
# ⚠️ SPENDS. --emulate-notebook supplies the two Jupyter-ambient defaults
# from 8.1 so the run measures the GOLD rather than the environment; drop it to
# measure the environment dependency instead. Teardown runs in a finally, and
# --sweep afterwards deletes anything that did not exist before the run.
!python live_gold_batch.py --run 5 --emulate-notebook

In [ ]:
!python live_gold_batch.py --sweep